In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [3]:
import numpy as np

In [4]:
from simulators.model_family import NestedModelFamily
from simulators.benchmarks.ddms.ddm import DDM
from simulators.benchmarks.ddms.ddm_priors import ddm_baseline_priors

In [5]:
import bayesflow as bf

INFO:bayesflow:Using backend 'torch'
When using torch backend, we need to disable autograd by default to avoid excessive memory usage. Use

with torch.enable_grad():
    ...

in contexts where you need gradients (e.g. custom training loops).


In [32]:
class DDMModelFamilyBF:

    def __init__(self):
        self.model_family = NestedModelFamily(
            model=DDM(),
            prior_fun=ddm_baseline_priors(),
            mask_randomizer_kwargs=dict(
                free_intrinsics=["v", "a", "tau", "s_v", "s_tau"],
                fixed_intrinsics=[],
                fixed_values={}
            )
    )

    def sample(self, batch_size, num_obs=500, flatten_param_outputs=False, **kwargs):

        if isinstance(batch_size, tuple):
             batch_size = batch_size[0]

        sample_kwargs = {
            "max_num_regressors": 0,
            "max_num_categories": 0
        }

        samples = self.model_family.batch_sample(
            batch_size=batch_size,
            num_obs=num_obs,
            flatten_param_outputs=flatten_param_outputs,
            **sample_kwargs,
            **kwargs
        )

        rts = samples["sim_data"]["rts"]
        choices = samples["sim_data"]["choices"]
        params = samples["param_matrices"]

        return {"rts": rts, "choices": choices, "params": params}

In [33]:
model_family = DDMModelFamilyBF()

In [34]:
ddm_samples = model_family.sample(4, num_obs=500)
for k, v in ddm_samples.items():
    print(k, v.shape if isinstance(v, np.ndarray) else (v.keys() if isinstance(v, dict) else v))

rts (4, 500, 1)
choices (4, 500, 1)
params (4, 1, 5)


In [35]:
adapter = (
    bf.Adapter()
    .convert_dtype("float64", "float32")
    .concatenate(["rts", "choices"], into="summary_variables")
    .squeeze("params", axis=1)
    .rename("params", "inference_variables")
)

In [36]:
adapted = adapter(model_family.sample(4))

In [37]:
for k, v in adapted.items():
    print(k, v.shape)

summary_variables (4, 500, 2)
inference_variables (4, 5)


In [38]:
summary_net = bf.networks.SetTransformer()
inference_net = bf.networks.FlowMatching()

In [39]:
workflow = bf.BasicWorkflow(
    simulator=model_family,
    adapter=adapter,
    inference_network=inference_net,
    summary_network=summary_net,
    checkpoint_filepath=f"../checkpoints/model_family_bf_intercept_only"
)

In [41]:
history = workflow.fit_online(
    epochs=50,
    steps_per_epoch=200,
    batch_size=32
)

INFO:bayesflow:Fitting on dataset instance of OnlineDataset.


Epoch 1/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 7s 36ms/step - loss: 1.0087
Epoch 2/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 38ms/step - loss: 1.0561
Epoch 3/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - loss: 1.0794
Epoch 4/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 43ms/step - loss: 0.9895
Epoch 5/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - loss: 0.9581
Epoch 6/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 49ms/step - loss: 0.9631
Epoch 7/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - loss: 0.9015
Epoch 8/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - loss: 0.8846 
Epoch 9/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 10s 52ms/step - loss: 0.8713 
Epoch 10/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 11s 55ms/step - loss: 0.8617 
Epoch 11/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 12s 60ms/step - loss: 0.8441 
Epoch 12/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 13s 66ms/step - loss: 0.8254 
Epoch 13/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 15s 75ms/step - loss: 0.8397 
Epoch 14/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 16s 79ms/step - loss: 0.8089 
Epoch 15/50
200/200 ━━━━━━━

INFO:bayesflow:Training is now finished.
            You can find the trained approximator at '../checkpoints/model_family_bf_intercept_only/model.model.keras'.
            To load it, use approximator = keras.saving.load_model(...).
